In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import ss2tf, tf2sos, dlti, dimpulse
import pyFDN


def isAlmostZero(x, tol=1e-12):
    return np.all(np.abs(x) < tol)


# ---------------- Example run ---------------- #
np.random.seed(1)

# Lossless feedback matrix
N = 3
m = np.array([13, 19, 23])
g = 0.9
A = pyFDN.random_orthogonal(N) @ np.diag(g ** m)
b = np.random.randn(N, 1)
c = np.random.randn(1, N)
d = np.random.randn(1, 1)

# State-space transition matrix
AA, bb, cc, dd = pyFDN.dss_to_ss(m, A, b, c, d)

# Compare impulse response
impulseResponseLength = 100

# Transfer function from state-space
num, den = ss2tf(AA, bb, cc, dd)
system = dlti(num, den)
t, irStateSpace_TF = dimpulse(system, n=impulseResponseLength)
irStateSpace_TF = np.squeeze(irStateSpace_TF)

# SOS representation
sos = tf2sos(num, den)
# Compute impulse response for SOS by converting to TF first
irStateSpace_SOS = np.zeros_like(irStateSpace_TF)
for section in sos:
    b_s, a_s = section[:3], section[3:]
    system_s = dlti(b_s, a_s)
    _, ir_s = dimpulse(system_s, n=impulseResponseLength)
    irStateSpace_SOS += np.squeeze(ir_s)
    
# Delay state-space impulse response
irDelayStateSpace = pyFDN.dss_to_impz(impulseResponseLength, m, A, b, c, d, input_type='mergeInput') # TODO: change to splitInput

# Plot
plt.figure()
plt.plot(irStateSpace_TF, label='State Space TF')
plt.plot(irStateSpace_SOS + 2, label='State Space SOS')
plt.plot(irDelayStateSpace + 4, label='Delay State Space')
plt.xlabel("Time [samples]")
plt.ylabel("Amplitude [linear]")
plt.grid(True)
plt.legend()
plt.show()

# ---------------- Tests ---------------- #
# All eigenvalues lie on the circle with radius g
e = np.linalg.eigvals(AA)
print(f"The pole magnitudes are between {np.min(np.abs(e))} and {np.max(np.abs(e))}")
assert isAlmostZero(np.abs(e) - g)

# Transfer function matches delay state-space
assert isAlmostZero(irStateSpace_TF - irDelayStateSpace, tol=0.001)

# SOS matches delay state-space
assert isAlmostZero(irStateSpace_SOS - irDelayStateSpace, tol=1e-12)